# Aula 05 — Rate limits na prática com Databricks AI Gateway

**Disciplina:** Deploy de Agentes com LLMs · **Autor:** Luiz Fernando Braz

No SAIBA MAIS vimos que um agente em produção é uma *torneira aberta de tokens*: um *loop*
descontrolado — ou um ataque de negação de serviço — pode multiplicar a conta da noite para
o dia. A defesa de infraestrutura para isso é o **rate limit**: o **AI Gateway** conta as
requisições por minuto de cada endpoint (ou usuário) e, ao estourar o limite, devolve um
**HTTP 429 (*Too Many Requests*)** em vez de chamar o modelo.

Aqui reaproveitamos os **mesmos dois endpoints** da aula servindo o `gpt-oss-20b`:

| Endpoint | Rate limit | O que esperamos |
|----------|-----------|------------------|
| `gpt-oss-20b-no-guardrails` | **nenhum** | 10 requisições passam, todas OK |
| `gpt-oss-20b-with-guardrails` | **5 requisições/minuto** | as 5 primeiras passam; da 6ª em diante, **HTTP 429** |

> **A proteção mora na infraestrutura, não no código.** O rate limit é ligado **na interface
> do Databricks**, no endpoint *with-guardrails*: em **Serving > (endpoint) > AI Gateway >
> Rate limits**, defina **5 queries por minuto**. Nosso código é o mesmo para os dois
> endpoints — só muda o nome. Para gastar **pouquíssimos tokens**, as respostas são
> minúsculas: só queremos contar requisições que *passam* vs. requisições *bloqueadas*.


## 1. Instalação e configuração

Os endpoints de *Model Serving* do Databricks expõem uma **API compatível com OpenAI**, então
falamos com eles pelo cliente **`ChatOpenAI`** (`langchain-openai`) — o mesmo do notebook de
guardrails.

In [ ]:
%pip install -q langchain-openai python-dotenv

# No Databricks, reinicia o Python para carregar as libs (ignorado fora do Databricks).
try:
    dbutils.library.restartPython()
except NameError:
    pass

### Credenciais e endpoints

Mesma configuração do notebook de guardrails: **host** e **token** do Databricks.

- **Dentro do Databricks:** vêm automaticamente do contexto do notebook.
- **Localmente:** preencha `DATABRICKS_HOST` e `DATABRICKS_TOKEN` no arquivo `aula-05/.env`.

In [ ]:
import os
from dotenv import load_dotenv

# Procura o .env na pasta da aula (útil ao rodar localmente, fora do Databricks).
for caminho in (".env", "../.env", "aula-05/.env"):
    if os.path.exists(caminho):
        load_dotenv(caminho)
        print(f"Variáveis carregadas de: {caminho}")
        break

DATABRICKS_HOST = os.getenv("DATABRICKS_HOST")
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")

# Rodando DENTRO do Databricks: host e token saem automaticamente do contexto do notebook.
if not (DATABRICKS_HOST and DATABRICKS_TOKEN):
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        DATABRICKS_HOST = DATABRICKS_HOST or ctx.apiUrl().get()
        DATABRICKS_TOKEN = DATABRICKS_TOKEN or ctx.apiToken().get()
        print("Credenciais obtidas do contexto do Databricks.")
    except NameError:
        pass

assert DATABRICKS_HOST and DATABRICKS_TOKEN, (
    "Defina DATABRICKS_HOST e DATABRICKS_TOKEN no aula-05/.env (fora do Databricks)."
)

# API compatível com OpenAI do Unity AI Gateway. O `model` é o NOME do endpoint.
BASE_URL = DATABRICKS_HOST.rstrip("/") + "/ai-gateway/mlflow/v1"
ENDPOINT_SEM = os.getenv("ENDPOINT_SEM_GUARDRAILS", "gpt-oss-20b-no-guardrails")
ENDPOINT_COM = os.getenv("ENDPOINT_COM_GUARDRAILS", "gpt-oss-20b-with-guardrails")

print("Base URL:", BASE_URL)
print("Endpoint SEM rate limit:", ENDPOINT_SEM)
print("Endpoint COM rate limit:", ENDPOINT_COM)

## 2. Clientes LangChain enxutos (poucos tokens)

O código é **idêntico** para os dois endpoints — muda só o nome. Limitamos a resposta a
pouquíssimos tokens (`max_tokens=16`): não nos interessa *o que* o modelo responde, só **se a
requisição passou ou foi bloqueada** pelo rate limit.

In [ ]:
import warnings
# gpt-oss é um modelo de raciocínio: devolve `content` como LISTA de blocos (reasoning +
# texto), e o Pydantic (via LangChain) espera `str`, emitindo um aviso de serialização a
# cada chamada. É inofensivo e não afeta o resultado — silenciamos só esse aviso para
# manter a saída do notebook limpa.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

def criar_cliente_leve(nome_endpoint: str) -> ChatOpenAI:
    return ChatOpenAI(
        model=nome_endpoint,        # no Databricks, o "model" é o NOME do endpoint
        base_url=BASE_URL,
        api_key=DATABRICKS_TOKEN,
        temperature=0,
        extra_body={"max_tokens": 16},  # respostas curtíssimas -> gasto mínimo de tokens
    )

llm_sem = criar_cliente_leve(ENDPOINT_SEM)   # sem rate limit
llm_com = criar_cliente_leve(ENDPOINT_COM)   # rate limit de 5/min (ligado na UI)

## 3. Uma rajada de requisições

`disparar()` envia o **mesmo prompt minúsculo** várias vezes seguidas e classifica cada
resposta. O segredo está no `try/except`: ao estourar o rate limit, o gateway responde
**HTTP 429** e o `ChatOpenAI` levanta uma exceção — nós a capturamos e marcamos a requisição
como **bloqueada**, em vez de quebrar o notebook.

In [ ]:
def disparar(llm, n_requisicoes: int, etiqueta: str) -> None:
    print("=" * 64)
    print(f"{etiqueta}: disparando {n_requisicoes} requisições seguidas")
    print("=" * 64)
    ok = bloqueadas = erros = 0
    for i in range(1, n_requisicoes + 1):
        try:
            llm.invoke([HumanMessage("Responda apenas: ok")])
            ok += 1
            print(f"  req {i:>2}: OK")
        except Exception as e:
            msg = str(e)
            eh_429 = getattr(e, "status_code", None) == 429 or "429" in msg or "Too Many Requests" in msg
            if eh_429:
                bloqueadas += 1
                print(f"  req {i:>2}: BLOQUEADA (HTTP 429 - rate limit)")
            else:
                erros += 1
                print(f"  req {i:>2}: ERRO - {msg[:80]}")
    print(f"\nResumo {etiqueta}: {ok} OK | {bloqueadas} bloqueadas (429) | {erros} erros")

## 4. Endpoint SEM rate limit — 10 requisições passam

No endpoint *no-guardrails* não há limite configurado: as 10 requisições devem retornar
**OK**.

In [ ]:
disparar(llm_sem, 10, "SEM rate limit (no-guardrails)")

## 5. Endpoint COM rate limit (5/min) — bloqueio após a 5ª

No endpoint *with-guardrails*, com **5 requisições/minuto** ligadas na UI, as **5 primeiras**
passam e, da **6ª em diante**, o gateway devolve **HTTP 429**.

> **Atenção à janela de 1 minuto:** o limite é por minuto. Se você reexecutar esta célula
> logo em seguida, a janela ainda pode estar saturada (tudo bloqueado) — espere ~60 s para o
> contador zerar.

In [ ]:
disparar(llm_com, 10, "COM rate limit 5/min (with-guardrails)")